# 16 — BocaSud residual investigation (v02)

**Question:** v02 EDITO results show BocaSud RMSE 0.106 m / Willmott 0.66 — markedly worse than BocaNord (0.07 / 0.85) and AltaVilaEst (0.08 / 0.78). Why?

**Three suspects:**
1. **Inlet-specific:** mesh under-resolves the southern channel; tide propagation damped → low amplitude + low mean
2. **Wrong replacement cell:** when v02 moved BS to an always-wet cell, we may have landed in a hydrodynamically different sub-basin
3. **Wind blending under-weights the south:** AE + Mulino stations are northern; IDW gives less weight to BS area → BS still ERA5-dominated (under-predicted wind)

**Phase 1 (this notebook):** all checks reuse the existing v02 `_map.nc` / `_his.nc`. No re-run.

**Run context:** designed for the EDITO JupyterLab (streams from S3 via s3fs). Same setup as notebook 15.

## 1. Setup — packages + S3 access

In [ ]:
import importlib, subprocess, sys
for pkg in ['xarray', 's3fs', 'h5netcdf', 'matplotlib', 'pandas', 'utide']:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import os, io, numpy as np, pandas as pd, xarray as xr, s3fs, utide
import matplotlib.pyplot as plt

BUCKET = 'oidc-cmartinsjr'
fs = s3fs.S3FileSystem(
    key=os.environ['AWS_ACCESS_KEY_ID'],
    secret=os.environ['AWS_SECRET_ACCESS_KEY'],
    token=os.environ.get('AWS_SESSION_TOKEN'),
    client_kwargs={'endpoint_url': os.environ.get('AWS_S3_ENDPOINT', 'https://minio.dive.edito.eu')},
)

his_key = next(p for p in fs.ls(f'{BUCKET}/DFM_OUTPUT/') if p.endswith('_his.nc'))
map_key = next(p for p in fs.ls(f'{BUCKET}/DFM_OUTPUT/') if p.endswith('_map.nc'))
print(f'his: {his_key}\nmap: {map_key}')

In [ ]:
# his.nc — load fully into memory (small, ~30 MB)
with fs.open(his_key, 'rb') as f:
    ds_his = xr.open_dataset(io.BytesIO(f.read()), engine='h5netcdf')

# Decode station names
stations = [s.decode().strip() if isinstance(s, bytes) else str(s).strip()
            for s in ds_his.station_name.values]
print('Stations:')
for i, s in enumerate(stations):
    print(f'  [{i:2d}] {s}')

# Find BocaSud index
idx_bs = next(i for i, s in enumerate(stations) if 'BocaSud' in s or 'Bocasud' in s.lower())
idx_bn = next(i for i, s in enumerate(stations) if 'BocaNord' in s or 'Bocanord' in s.lower())
idx_ae = next(i for i, s in enumerate(stations) if 'AltaVila' in s or 'Altavila' in s.lower())
print(f'\nBN idx={idx_bn} at ({float(ds_his.station_x_coordinate.isel(station=idx_bn)):.4f}, {float(ds_his.station_y_coordinate.isel(station=idx_bn)):.4f})')
print(f'BS idx={idx_bs} at ({float(ds_his.station_x_coordinate.isel(station=idx_bs)):.4f}, {float(ds_his.station_y_coordinate.isel(station=idx_bs)):.4f})')
print(f'AE idx={idx_ae} at ({float(ds_his.station_x_coordinate.isel(station=idx_ae)):.4f}, {float(ds_his.station_y_coordinate.isel(station=idx_ae)):.4f})')

## 2. Load observed BS time series

QC'd in-situ data was pushed to S3 under `CODE/data/processed/`. Pull the BS series.

In [ ]:
import boto3
s3 = boto3.client('s3',
    endpoint_url=os.environ.get('AWS_S3_ENDPOINT', 'https://minio.dive.edito.eu'),
    aws_access_key_id=os.environ['AWS_ACCESS_KEY_ID'],
    aws_secret_access_key=os.environ['AWS_SECRET_ACCESS_KEY'],
    aws_session_token=os.environ.get('AWS_SESSION_TOKEN'),
)

obs_files = {
    'BocaSud':     'CODE/data/processed/wl_BocaSud_10min_UTC.csv',
    'BocaNord':    'CODE/data/processed/wl_BocaNord_10min_UTC.csv',
    'AltaVilaEst': 'CODE/data/processed/wl_AltavilaEst_10min_UTC.csv',
}
obs = {}
for name, key in obs_files.items():
    body = s3.get_object(Bucket=BUCKET, Key=key)['Body'].read()
    df = pd.read_csv(io.BytesIO(body), index_col=0, parse_dates=True)
    df.columns = ['wl_obs']
    obs[name] = df
    print(f'{name}: {len(df)} samples, {df.index[0]} → {df.index[-1]}')

## 3. Time-series overlay — visual diagnosis

First look: is the BS residual (a) constant bias, (b) damped amplitude, (c) phase shift, or (d) episodic divergence?

In [ ]:
def model_series(idx):
    s = ds_his.waterlevel.isel(station=idx).to_pandas()
    return s.rename('wl_model')

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
for ax, (name, idx) in zip(axes, [('BocaNord', idx_bn), ('BocaSud', idx_bs), ('AltaVilaEst', idx_ae)]):
    mod = model_series(idx)
    df = pd.concat([mod, obs[name]['wl_obs']], axis=1).dropna()
    df = df[df.index >= df.index[0] + pd.Timedelta(hours=12)]   # drop spinup
    ax.plot(df.index, df['wl_obs'],   '-', lw=1.0, label='Observed', color='black')
    ax.plot(df.index, df['wl_model'], '-', lw=0.9, label='v02 model', color='tab:blue', alpha=0.85)
    bias = (df['wl_model'] - df['wl_obs']).mean()
    rmse = ((df['wl_model'] - df['wl_obs'])**2).mean()**0.5
    ax.set_title(f'{name}   bias={bias:+.3f} m   RMSE={rmse:.3f} m')
    ax.set_ylabel('WL (m)'); ax.grid(alpha=0.3); ax.legend(loc='upper right', fontsize=9)
axes[-1].set_xlabel('Time (UTC)')
plt.tight_layout(); plt.show()

In [ ]:
# Zoom on a 2-day window to inspect tidal cycles in detail
fig, ax = plt.subplots(figsize=(13, 4))
name, idx = 'BocaSud', idx_bs
mod = model_series(idx)
df = pd.concat([mod, obs[name]['wl_obs']], axis=1).dropna()
t0 = df.index[0] + pd.Timedelta(days=2)
df_zoom = df[(df.index >= t0) & (df.index <= t0 + pd.Timedelta(days=2))]
ax.plot(df_zoom.index, df_zoom['wl_obs'],   '-o', ms=2, label='Observed', color='black')
ax.plot(df_zoom.index, df_zoom['wl_model'], '-o', ms=2, label='v02 model', color='tab:blue')
ax.set_title('BocaSud — 48 h zoom (look for amplitude flatness and phase shift)')
ax.set_ylabel('WL (m)'); ax.grid(alpha=0.3); ax.legend(); plt.tight_layout(); plt.show()

## 4. Tidal harmonic decomposition

If model M2/S2 amplitudes are systematically lower than observed, the issue is mesh/friction at the inlet (waves are damped). If amplitudes match but phases are off, the issue is timing (resolution / propagation speed).

In [ ]:
def harmonics(series, lat=37.86):
    """Run utide.solve on an evenly sampled series, return amplitude/phase for major constituents."""
    s = series.dropna()
    t = s.index.to_julian_date().values
    coef = utide.solve(t, s.values, lat=lat,
                       method='ols', conf_int='linear', verbose=False,
                       constit=['M2', 'S2', 'K1', 'O1', 'N2', 'P1'])
    return pd.DataFrame({'name': coef.name, 'A': coef.A, 'g': coef.g}).set_index('name')

df_bs = pd.concat([model_series(idx_bs), obs['BocaSud']['wl_obs']], axis=1).dropna()
h_obs = harmonics(df_bs['wl_obs'])
h_mod = harmonics(df_bs['wl_model'])

comp = pd.concat([
    h_obs.add_suffix('_obs'),
    h_mod.add_suffix('_mod'),
], axis=1)
comp['A_ratio'] = comp['A_mod'] / comp['A_obs']
comp['g_diff_deg'] = ((comp['g_mod'] - comp['g_obs'] + 180) % 360) - 180
comp[['A_obs', 'A_mod', 'A_ratio', 'g_obs', 'g_mod', 'g_diff_deg']].round(3)

**Interpretation rules:**
- `A_ratio < 0.7` for M2/S2 → significant amplitude damping, points to inlet bathymetry/friction (suspect 1)
- `|g_diff_deg| > 20°` for M2 → phase lag, points to coarse mesh propagation (suspect 1)
- `A_ratio ≈ 1` and `|g_diff_deg| < 5°` → tides are fine; bias is in the *mean* level → suspect 2 (cell location) or external forcing

## 5. Old cell vs new cell — isolate the replacement effect

v02 moved the BS obs to a deeper always-wet cell (notebook 10). Sample the *original* BS coordinate inside the same v02 `map.nc` and compare both as if observed there. If the old cell shows a different signal than the new cell, the replacement choice is part of the residual.

In [ ]:
# Open map.nc lazily
fobj = fs.open(map_key, mode='rb')
ds_map = xr.open_dataset(fobj, engine='h5netcdf', chunks={'time': 50})
fx = ds_map['mesh2d_face_x'].values
fy = ds_map['mesh2d_face_y'].values
min_depth = ds_map['mesh2d_waterdepth'].min(dim='time').compute().values

# Original BocaSud (Model B, before replacement)
BS_OLD_LON, BS_OLD_LAT = 12.4490, 37.8470
# Current BS in v02 his.nc (replacement cell coordinate)
BS_NEW_LON = float(ds_his.station_x_coordinate.isel(station=idx_bs))
BS_NEW_LAT = float(ds_his.station_y_coordinate.isel(station=idx_bs))

def nearest_face(lon, lat):
    d2 = (fx - lon)**2 + (fy - lat)**2
    i = int(np.argmin(d2))
    return i, fx[i], fy[i], min_depth[i]

i_old, x_old, y_old, d_old = nearest_face(BS_OLD_LON, BS_OLD_LAT)
i_new, x_new, y_new, d_new = nearest_face(BS_NEW_LON, BS_NEW_LAT)

print(f'OLD BS at ({BS_OLD_LON:.4f}, {BS_OLD_LAT:.4f})')
print(f'  → face {i_old} at ({x_old:.4f}, {y_old:.4f}), min depth = {d_old:.3f} m')
print(f'NEW BS at ({BS_NEW_LON:.4f}, {BS_NEW_LAT:.4f})')
print(f'  → face {i_new} at ({x_new:.4f}, {y_new:.4f}), min depth = {d_new:.3f} m')

wl_old = ds_map['mesh2d_s1'].isel(mesh2d_nFaces=i_old).compute().to_pandas()
wl_new = ds_map['mesh2d_s1'].isel(mesh2d_nFaces=i_new).compute().to_pandas()

In [ ]:
# Compare both sampled cells against observation
df_cmp = pd.concat([
    wl_old.rename('wl_old_cell'),
    wl_new.rename('wl_new_cell'),
    obs['BocaSud']['wl_obs'],
], axis=1).dropna()
df_cmp = df_cmp[df_cmp.index >= df_cmp.index[0] + pd.Timedelta(hours=12)]

def metrics(mod, obs):
    bias = (mod - obs).mean()
    rmse = ((mod - obs)**2).mean()**0.5
    var_obs = ((obs - obs.mean())**2).mean()
    skill = 1 - ((mod - obs)**2).sum() / ((np.abs(mod - obs.mean()) + np.abs(obs - obs.mean()))**2).sum()
    return bias, rmse, skill

for col in ['wl_old_cell', 'wl_new_cell']:
    b, r, s = metrics(df_cmp[col], df_cmp['wl_obs'])
    print(f'{col:14s}  bias={b:+.4f} m   RMSE={r:.4f} m   Willmott={s:.3f}')

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(df_cmp.index, df_cmp['wl_obs'], '-', lw=1.0, label='Observed', color='black')
ax.plot(df_cmp.index, df_cmp['wl_old_cell'], '-', lw=0.8, label='OLD BS cell (Model B coord)', color='tab:red', alpha=0.8)
ax.plot(df_cmp.index, df_cmp['wl_new_cell'], '-', lw=0.8, label='NEW BS cell (v02 replacement)', color='tab:blue', alpha=0.8)
ax.set_title('BocaSud: observed vs old/new model cells (same v02 run)')
ax.set_ylabel('WL (m)'); ax.grid(alpha=0.3); ax.legend(); plt.tight_layout(); plt.show()

## 6. Search for a better replacement cell

Sample all always-wet cells within 500 m of the original BS coordinate. Score each against observation. Pick the best.

In [ ]:
SEARCH_RADIUS_DEG = 0.005   # ~500 m at this latitude
MIN_DEPTH = 0.15

d2 = (fx - BS_OLD_LON)**2 + (fy - BS_OLD_LAT)**2
candidates = np.where((d2 < SEARCH_RADIUS_DEG**2) & (min_depth > MIN_DEPTH))[0]
print(f'{len(candidates)} always-wet candidate cells within 500 m of original BS')

In [ ]:
# Score each candidate against obs. To stay under quota, sample at most ~50 cells.
if len(candidates) > 50:
    rng = np.random.default_rng(42)
    candidates = rng.choice(candidates, size=50, replace=False)

obs_bs = obs['BocaSud']['wl_obs']
scores = []
print('Sampling candidates from map.nc (this can take a few minutes)...')
for c in candidates:
    wl = ds_map['mesh2d_s1'].isel(mesh2d_nFaces=int(c)).compute().to_pandas()
    df = pd.concat([wl.rename('m'), obs_bs], axis=1).dropna()
    df = df[df.index >= df.index[0] + pd.Timedelta(hours=12)]
    if len(df) < 100:
        continue
    b, r, s = metrics(df['m'], df['wl_obs'])
    dist_m = np.sqrt((fx[c] - BS_OLD_LON)**2 + (fy[c] - BS_OLD_LAT)**2) * 111000 * np.cos(np.radians(BS_OLD_LAT))
    scores.append({
        'face': int(c), 'lon': float(fx[c]), 'lat': float(fy[c]),
        'min_depth': float(min_depth[c]), 'dist_m': float(dist_m),
        'bias': b, 'rmse': r, 'willmott': s,
    })
scores_df = pd.DataFrame(scores).sort_values('willmott', ascending=False).reset_index(drop=True)
print(f'\nTop 10 candidates by Willmott skill:')
scores_df.head(10).round(4)

In [ ]:
# Map view of candidates colored by skill
fig, ax = plt.subplots(figsize=(7, 8))
sc = ax.scatter(scores_df['lon'], scores_df['lat'], c=scores_df['willmott'],
                cmap='viridis', s=60, edgecolor='k', linewidth=0.4)
ax.scatter([BS_OLD_LON], [BS_OLD_LAT], color='red', marker='x', s=200, linewidth=3, label='original BS coord')
ax.scatter([BS_NEW_LON], [BS_NEW_LAT], color='blue', marker='+', s=200, linewidth=3, label='current v02 cell')
best = scores_df.iloc[0]
ax.scatter([best['lon']], [best['lat']], facecolors='none', edgecolors='lime', s=240, linewidth=2.5, label=f'best (Willmott={best["willmott"]:.3f})')
plt.colorbar(sc, ax=ax, label='Willmott skill at BS')
ax.set_aspect(1/np.cos(np.radians(37.86)))
ax.set_xlabel('Lon'); ax.set_ylabel('Lat')
ax.set_title('BS candidate cells within 500 m')
ax.legend(); plt.tight_layout(); plt.show()

## 7. Verdict template

Fill this in based on the cells above:

| Test | Result | Implication |
|------|--------|-------------|
| Time-series shape (cell 7-8) | [bias / amplitude flat / phase lag / episodic] | [...] |
| M2 A_ratio | [value] | [<0.7 → inlet damping] |
| M2 phase diff | [value] | [>20° → propagation issue] |
| Old vs new cell (cell 14) | [old better / new better / same] | [if old better → revisit replacement] |
| Best candidate Willmott (cell 17) | [value] | [if much > current 0.66 → re-locate BS in v03] |
| Best candidate distance from original | [value] m | [closer = less concerning] |

**Recommended action depending on outcome:**
- **Cell relocation gain > 0.05 in Willmott:** update `Stagnone_dxy01_15m_obs.xyn` for v03 with the better cell; no model rerun needed for the metric (just re-post-process)
- **Amplitude damping (M2 ratio < 0.8):** mesh refinement or local Manning reduction at the southern channel — needs v03 rerun
- **Bias only, amplitude OK:** likely external forcing (wind blend under-weighting south, or evap missing) — fix in v03
- **Phase lag with intact amplitude:** propagation speed in coarse cells — mesh refinement